In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")
from core.startup import init
engine, memory = init()

In [ ]:
!pip install sentence-transformers -q

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
model.save("/workspace/models/results/all-MiniLM-L6-v2")
print("Saved.")
del model

In [ ]:
!cp /workspace/Projects/cultivated-learning/engine/inference.py /workspace/Projects/cultivated-learning/engine/inference_backup.py
print("Backup saved.")

In [ ]:
with open("/workspace/Projects/cultivated-learning/engine/inference.py", "w") as f:
    f.write('''import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer


class InferenceEngine:
    """Wrapper around the base LLM. All model interaction goes through here."""
    
    def __init__(self, model_path, max_context=4096, 
                 embedding_model_path="/workspace/models/results/all-MiniLM-L6-v2"):
        self.model_path = model_path
        self.max_context = max_context
        self.embedding_model_path = embedding_model_path
        self.model = None
        self.tokenizer = None
        self.device = None
        self.embedding_model = None
    
    def load(self):
        # Load Mistral for generation
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_path, local_files_only=True
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_path,
            dtype=torch.float16,
            device_map="auto",
            local_files_only=True
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.device = self.model.device
        
        # Load dedicated embedding model
        self.embedding_model = SentenceTransformer(
            self.embedding_model_path, device=str(self.device)
        )
        
        vram = torch.cuda.memory_allocated(0) / 1e9
        print(f"Loaded {self.model_path}")
        print(f"Loaded embedding model: {self.embedding_model_path}")
        print(f"  VRAM: {vram:.2f} GB")
        print(f"  Embedding dim: {self.embedding_model.get_sentence_embedding_dimension()}")
        print(f"  Max context: {self.max_context} tokens")
        return self
    
    def count_tokens(self, text):
        return len(self.tokenizer.encode(text, add_special_tokens=False))
    
    def generate(self, prompt, max_new_tokens=512, temperature=0.7, top_p=0.9):
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_context - max_new_tokens
        ).to(self.device)
        
        with torch.no_grad():
            output_ids = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                do_sample=True,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
            )
        
        prompt_len = inputs["input_ids"].shape[-1]
        new_tokens = output_ids[0][prompt_len:]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        
        del inputs, output_ids
        torch.cuda.empty_cache()
        
        return response
    
    def generate_structured(self, prompt, max_new_tokens=512, temperature=0.3):
        return self.generate(prompt, max_new_tokens=max_new_tokens, temperature=temperature, top_p=0.95)
    
    def get_embedding(self, text):
        """Generate embedding using dedicated sentence-transformer model (384-dim)."""
        embedding = self.embedding_model.encode(text, normalize_embeddings=True)
        return embedding
    
    def get_embedding_dimension(self):
        """Return the dimensionality of the embedding model."""
        return self.embedding_model.get_sentence_embedding_dimension()
''')

print("inference.py updated.")

In [ ]:
import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")

from engine.inference import InferenceEngine

engine = InferenceEngine("/workspace/models/results/Mistral-7B-Instruct-v0.3")
engine.load()

In [ ]:
with open("/workspace/Projects/cultivated-learning/core/memory_store.py", "r") as f:
    print(f.read())

In [ ]:
from core.memory_store import MemoryStore, MemoryUnit, MemoryType

# Load with old data
memory = MemoryStore(
    persist_dir="/workspace/Projects/cultivated-learning/data/memory_db",
    engine=engine
)

# Export existing memories before wiping
old_data = memory.collection.get()
print(f"Backing up {len(old_data['ids'])} memories...")

# Save backup
import json
backup = []
for i in range(len(old_data['ids'])):
    backup.append({
        "id": old_data["ids"][i],
        "document": old_data["documents"][i],
        "metadata": old_data["metadatas"][i]
    })

with open("/workspace/Projects/cultivated-learning/data/memory_backup.json", "w") as f:
    json.dump(backup, f, indent=2)
print("Backup saved to memory_backup.json")

# Delete old collection and recreate
memory.client.delete_collection("cultivated_memory")
memory.collection = memory.client.get_or_create_collection(
    name="cultivated_memory",
    metadata={"hnsw:space": "cosine"}
)
print(f"Collection rebuilt. Count: {memory.collection.count()}")

# Re-embed and store each memory
for item in backup:
    mem = MemoryUnit.from_chroma(
        id=item["id"],
        document=item["document"],
        metadata=item["metadata"]
    )
    memory.store(mem)  # Will re-embed with 384-dim model
    print(f"  Re-embedded: {mem.content[:60]}...")

print(f"\nDone. {memory.collection.count()} memories re-embedded at 384-dim.")

In [ ]:
results = memory.retrieve("What project is the user working on?", top_k=3)
for r in results:
    print(f"[{r.salience_score:.2f}] {r.content[:80]}...")

In [ ]:
results = memory.retrieve("Contact Front video game", top_k=5)
for r in results:
    print(f"[sal:{r.salience_score:.2f}] {r.content[:80]}...")

In [ ]:
# Back up first
!cp /workspace/Projects/cultivated-learning/core/memory_store.py /workspace/Projects/cultivated-learning/core/memory_store_backup.py
print("Backup saved.")

In [ ]:
# Check what fields ChromaDB gives us
raw = memory.collection.query(
    query_embeddings=[engine.get_embedding("Contact Front video game").tolist()],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)
for i in range(len(raw["ids"][0])):
    doc = raw["documents"][0][i][:60]
    dist = raw["distances"][0][i]
    sal = raw["metadatas"][0][i]["salience_score"]
    sim = 1 - dist  # cosine distance → similarity
    print(f"sim:{sim:.3f}  sal:{sal:.2f}  doc:{doc}...")

In [ ]:
with open("/workspace/Projects/cultivated-learning/core/memory_store.py", "r") as f:
    original = f.read()

# Replace the retrieve method
old_retrieve = '''    def retrieve(self, query_text, top_k=10, min_salience=0.1):
        if self.collection.count() == 0:
            return []

        if self.engine is not None:
            query_embedding = self.engine.get_embedding(query_text)
            results = self.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=min(top_k * 3, self.collection.count()),
            )
        else:
            results = self.collection.query(
                query_texts=[query_text],
                n_results=min(top_k * 3, self.collection.count()),
            )

        memories = []
        for i in range(len(results["ids"][0])):
            mem = MemoryUnit.from_chroma(
                id=results["ids"][0][i],
                document=results["documents"][0][i],
                metadata=results["metadatas"][0][i],
            )
            if mem.salience_score >= min_salience and mem.superseded_by is None:
                mem.last_accessed = time.time()
                mem.access_count += 1
                self._update_access(mem)
                memories.append(mem)

        memories.sort(key=lambda m: m.salience_score, reverse=True)
        return memories[:top_k]'''

new_retrieve = '''    def retrieve(self, query_text, top_k=10, min_salience=0.1,
                 similarity_weight=0.6, salience_weight=0.4):
        if self.collection.count() == 0:
            return []

        if self.engine is not None:
            query_embedding = self.engine.get_embedding(query_text)
            results = self.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=min(top_k * 3, self.collection.count()),
                include=["documents", "metadatas", "distances"],
            )
        else:
            results = self.collection.query(
                query_texts=[query_text],
                n_results=min(top_k * 3, self.collection.count()),
                include=["documents", "metadatas", "distances"],
            )

        scored_memories = []
        for i in range(len(results["ids"][0])):
            mem = MemoryUnit.from_chroma(
                id=results["ids"][0][i],
                document=results["documents"][0][i],
                metadata=results["metadatas"][0][i],
            )
            if mem.salience_score >= min_salience and mem.superseded_by is None:
                mem.last_accessed = time.time()
                mem.access_count += 1
                self._update_access(mem)
                similarity = 1 - results["distances"][0][i]  # cosine distance → similarity
                blended_score = (similarity * similarity_weight) + (mem.salience_score * salience_weight)
                scored_memories.append((mem, blended_score))

        scored_memories.sort(key=lambda x: x[1], reverse=True)
        return [m for m, s in scored_memories[:top_k]]'''

updated = original.replace(old_retrieve, new_retrieve)

with open("/workspace/Projects/cultivated-learning/core/memory_store.py", "w") as f:
    f.write(updated)

print("memory_store.py updated — retrieve() now blends similarity + salience.")

In [ ]:
# Reload the module
import importlib
import core.memory_store
importlib.reload(core.memory_store)
from core.memory_store import MemoryStore

memory = MemoryStore(
    persist_dir="/workspace/Projects/cultivated-learning/data/memory_db",
    engine=engine
)

results = memory.retrieve("What project is the user working on?", top_k=3)
for r in results:
    print(f"[{r.salience_score:.2f}] {r.content[:80]}...")

In [ ]:
results = memory.retrieve("What project is the user working on?", top_k=5)
for r in results:
    print(f"[{r.salience_score:.2f}] {r.content[:80]}...")

In [ ]:
import os
print(os.path.exists("/workspace/Projects/cultivated-learning/ui"))
print(os.listdir("/workspace/Projects/cultivated-learning/"))

In [ ]:
# Create UI directory
!mkdir -p /workspace/Projects/cultivated-learning/ui

# Backup nothing — new file
with open("/workspace/Projects/cultivated-learning/ui/app.py", "w") as f:
    f.write('''import sys
sys.path.insert(0, "/workspace/Projects/cultivated-learning")

import gradio as gr
from engine.inference import InferenceEngine
from core.memory_store import MemoryStore
from core.context_assembler import ContextAssembler
from core.interaction_loop import InteractionLoop

# Initialize
print("Loading models...")
engine = InferenceEngine("/workspace/models/results/Mistral-7B-Instruct-v0.3")
engine.load()

memory = MemoryStore(
    persist_dir="/workspace/Projects/cultivated-learning/data/memory_db",
    engine=engine
)

assembler = ContextAssembler(engine=engine, memory_store=memory)

loop = InteractionLoop(
    engine=engine,
    memory_store=memory,
    assembler=assembler,
    log_dir="/workspace/Projects/cultivated-learning/data/interaction_log"
)
print("Ready.")


def chat(message, history):
    """Send message through the full cognitive pipeline."""
    response = loop.chat(message)
    return response


def give_feedback(rating, correction):
    """Submit feedback on the last interaction."""
    rating = int(rating)
    corr = correction.strip() if correction.strip() else None
    loop.feedback(rating=rating, correction=corr)
    msg = f"Feedback recorded: rating {rating}"
    if corr:
        msg += f", correction stored"
    return msg


def get_status():
    """Show memory stats."""
    stats = memory.get_stats()
    status = loop.status()
    lines = [
        f"Interactions: {status.get(\'interaction_count\', \'?\')}",
        f"Memories: {stats[\'total\']}",
    ]
    if stats["total"] > 0:
        lines.append(f"By type: {stats[\'by_type\']}")
        lines.append(f"Avg salience: {stats[\'avg_salience\']:.3f}")
        lines.append(f"Range: {stats[\'min_salience\']:.3f} — {stats[\'max_salience\']:.3f}")
    return "\\n".join(lines)


with gr.Blocks(title="Cultivated Learning", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🌱 Cultivated Learning")
    gr.Markdown("*Frozen model. Growing mind.*")
    
    with gr.Tab("Chat"):
        chatbot = gr.ChatInterface(
            fn=chat,
            type="messages",
        )
    
    with gr.Tab("Feedback"):
        gr.Markdown("### Rate the last response")
        rating = gr.Slider(minimum=1, maximum=5, step=1, value=3, label="Rating (1-5)")
        correction = gr.Textbox(
            label="Correction (optional)", 
            placeholder="e.g. Be more concise next time"
        )
        fb_btn = gr.Button("Submit Feedback")
        fb_output = gr.Textbox(label="Result", interactive=False)C
        fb_btn.click(fn=give_feedback, inputs=[rating, correction], outputs=fb_output)
    
    with gr.Tab("Status"):
        status_output = gr.Textbox(label="System Status", interactive=False, lines=6)
        status_btn = gr.Button("Refresh")
        status_btn.click(fn=get_status, outputs=status_output)

app.launch(server_name="0.0.0.0", server_port=7860, share=False)
''')

print("ui/app.py created.")

In [ ]:
with open("/workspace/Projects/cultivated-learning/ui/app.py", "r") as f:
    content = f.read()

content = content.replace("server_port=7860", "server_port=7880")

with open("/workspace/Projects/cultivated-learning/ui/app.py", "w") as f:
    f.write(content)

print("Updated to port 7880.")

In [ ]:
with open("/workspace/Projects/cultivated-learning/ui/app.py", "r") as f:
    content = f.read()

content = content.replace(
    '''            type="messages",''',
    ''''''
)

with open("/workspace/Projects/cultivated-learning/ui/app.py", "w") as f:
    f.write(content)

print("Fixed.")